In [1]:
import pandas as pd

meta = pd.read_csv('/projects/bodymaps/Data/UCSFLLMOutputLarge27k.csv')

/local/psalvad2/875827/ipykernel_2533745/2667273785.py:3: DtypeWarning: Columns (11,19,21) have mixed types. Specify dtype option on import or set low_memory=False.
  meta = pd.read_csv('/projects/bodymaps/Data/UCSFLLMOutputLarge27k.csv')


In [2]:
meta

,Encrypted Patient MRN,Tumor ID,Organ,Tumor Type,Tumor Location,Tumor Size (mm),Tumor Attenuation,Type Certainty,DNN Answer,Report,...,Standardized Attenuation,Standardized Organ,Unknow Tumor Size,Standardized Location,BDMAP_ID,Patient Age,Patient Sex,no lesion,Index,Mapped Attenuation
0,AQPHRXY1,tumor 1,kidney,u,perinephric,5.0,low,u,Answer (template filled):\n\ntumor 1: type = U...,INDICATION:\r\nCT ABDOMEN/PELVIS WITH CONTRAST...,...,low,kidney,no,u,BDMAP_00038206,69.0,Female,False,NaN,NaN
1,x0rEVlgI,tumor 1,kidney,u,u,9.0,hypodense,u,Answer (template filled):\n\ntumor 1: type = U...,INDICATION:\r\nCT ABDOMEN/PELVIS WITH CONTRAST...,...,low,kidney,no,u,BDMAP_00041303,66.0,Female,False,NaN,NaN
2,x0rEVlgI,tumor 2,spleen,cyst,u,u,hypodense,high,Answer (template filled):\n\ntumor 1: type = U...,INDICATION:\r\nCT ABDOMEN/PELVIS WITH CONTRAST...,...,low,spleen,yes,NaN,BDMAP_00041303,66.0,Female,False,NaN,NaN
3,ZLND5q9x,tumor 1,liver,metastasis,right lobe,u,hypoattenuating,low,Answer (template filled):\n\ntumor 1: type = m...,INTERPRETATION OF OUTSIDE [ADDRESS] ABDOMEN AN...,...,low,liver,yes,segment 5 / segment 6 / segment 7 / segment 8,NaN,62.0,Female,False,NaN,NaN
4,ZLND5q9x,tumor 2,liver,metastasis,right lobe,u,hypoattenuating,low,Answer (template filled):\n\ntumor 1: type = m...,INTERPRETATION OF OUTSIDE [ADDRESS] ABDOMEN AN...,...,low,liver,yes,segment 5 / segment 6 / segment 7 / segment 8,NaN,62.0,Female,False,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96846,NaN,tumor 2,liver,metastasis,u,13.0,u,high,Answer (template filled):\n\ntumor 1: type = P...,INDICATION:\r\nCT ABDOMEN/PELVIS WITH AND WITH...,...,NaN,liver,yes,u,BDMAP_00085075,NaN,NaN,NaN,332.0,u
96847,NaN,tumor 3,liver,metastasis,u,multiple,u,high,Answer (template filled):\n\ntumor 1: type = P...,INDICATION:\r\nCT ABDOMEN/PELVIS WITH AND WITH...,...,NaN,liver,yes,u,BDMAP_00085075,NaN,NaN,NaN,332.0,u
96848,NaN,tumor 4,bone,metastasis,vertebral body of l1,22.0,u,high,Answer (template filled):\n\ntumor 1: type = P...,INDICATION:\r\nCT ABDOMEN/PELVIS WITH AND WITH...,...,NaN,bone,no,NaN,BDMAP_00085075,NaN,NaN,NaN,332.0,u
96849,NaN,tumor 5,bone,metastasis,right lower rib,11.0,u,high,Answer (template filled):\n\ntumor 1: type = P...,INDICATION:\r\nCT ABDOMEN/PELVIS WITH AND WITH...,...,NaN,bone,no,NaN,BDMAP_00085075,NaN,NaN,NaN,332.0,u


In [3]:
def fill_location(row):
    # Check if the 'Tumor Location' is NaN or empty; if so, return 'u'
    if pd.isna(row['Tumor Location']):
        return 'u'
    # Convert the tumor location to lowercase to perform a case-insensitive check
    tumor_lower = row['Tumor Location'].lower()
    if 'right' in tumor_lower:
        return 'right'
    elif 'left' in tumor_lower:
        return 'left'
    else:
        return 'u'

# Create a mask for rows where Standardized Organ is "Adrenal Gland" (case insensitive)
mask = meta['Standardized Organ'].str.lower() == 'adrenal gland'

# Apply the function only on rows matching the mask
meta.loc[mask, 'Standardized Location'] = meta.loc[mask].apply(fill_location, axis=1)


In [4]:
for org in ['lung','breast','femur']:
    # Create a mask for rows where Standardized Organ is "Adrenal Gland" (case insensitive)
    mask = meta['Standardized Organ'].str.lower() == org

    # Apply the function only on rows matching the mask
    meta.loc[mask, 'Standardized Location'] = meta.loc[mask].apply(fill_location, axis=1)

In [5]:
meta[meta['BDMAP_ID']=='BDMAP_00080875']

,Encrypted Patient MRN,Tumor ID,Organ,Tumor Type,Tumor Location,Tumor Size (mm),Tumor Attenuation,Type Certainty,DNN Answer,Report,...,Standardized Attenuation,Standardized Organ,Unknow Tumor Size,Standardized Location,BDMAP_ID,Patient Age,Patient Sex,no lesion,Index,Mapped Attenuation
94091,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT OF THE ABDOMEN AND PELVIS: [DATE].\r\n\r\nC...,...,NaN,NaN,NaN,NaN,BDMAP_00080875,45.0,Male,True,NaN,NaN


In [3]:

import numpy as np
#y = np.load('/projects/bodymaps/Data/UFO_27k_medformerNpz/BDMAP_00085207_gt.npz')['arr_0']
y = np.load('/projects/bodymaps/Pedro/data/UCSF_structured_report_subsegments_medformer_npz/BDMAP_00084817_gt.npz')['arr_0']



In [4]:
y.shape

(35, 435, 267, 403)

In [20]:
#labels:
pth = '/projects/bodymaps/Pedro/data/UFO_medformer_npy/list/label_names.yaml'
import yaml
with open(pth, 'r') as f:
    classes_UFO = yaml.load(f, Loader=yaml.SafeLoader)
    #sort--we sorted when saving in nii2npy.py
    classes_UFO = sorted(classes_UFO)

In [21]:
len(classes_UFO)

35

In [15]:
classes_UFO.index('spleen')

33

In [23]:
y[classes_UFO.index('adrenal_gland_left')].max()

1

In [24]:
# Assuming your DataFrame is named df
meta.drop("Unnamed: 0", axis=1, inplace=True)

KeyError: "['Unnamed: 0'] not found in axis"

In [8]:
meta

,Encrypted Patient MRN,Tumor ID,Organ,Tumor Type,Tumor Location,Tumor Size (mm),Tumor Attenuation,Type Certainty,DNN Answer,Report,...,Standardized Attenuation,Standardized Organ,Unknow Tumor Size,Standardized Location,BDMAP_ID,Patient Age,Patient Sex,no lesion,Index,Mapped Attenuation
0,AQPHRXY1,tumor 1,kidney,u,perinephric,5.0,low,u,Answer (template filled):\n\ntumor 1: type = U...,INDICATION:\r\nCT ABDOMEN/PELVIS WITH CONTRAST...,...,low,kidney,no,u,BDMAP_00038206,69.0,Female,False,NaN,NaN
1,x0rEVlgI,tumor 1,kidney,u,u,9.0,hypodense,u,Answer (template filled):\n\ntumor 1: type = U...,INDICATION:\r\nCT ABDOMEN/PELVIS WITH CONTRAST...,...,low,kidney,no,u,BDMAP_00041303,66.0,Female,False,NaN,NaN
2,x0rEVlgI,tumor 2,spleen,cyst,u,u,hypodense,high,Answer (template filled):\n\ntumor 1: type = U...,INDICATION:\r\nCT ABDOMEN/PELVIS WITH CONTRAST...,...,low,spleen,yes,NaN,BDMAP_00041303,66.0,Female,False,NaN,NaN
3,ZLND5q9x,tumor 1,liver,metastasis,right lobe,u,hypoattenuating,low,Answer (template filled):\n\ntumor 1: type = m...,INTERPRETATION OF OUTSIDE [ADDRESS] ABDOMEN AN...,...,low,liver,yes,segment 5 / segment 6 / segment 7 / segment 8,NaN,62.0,Female,False,NaN,NaN
4,ZLND5q9x,tumor 2,liver,metastasis,right lobe,u,hypoattenuating,low,Answer (template filled):\n\ntumor 1: type = m...,INTERPRETATION OF OUTSIDE [ADDRESS] ABDOMEN AN...,...,low,liver,yes,segment 5 / segment 6 / segment 7 / segment 8,NaN,62.0,Female,False,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96846,NaN,tumor 2,liver,metastasis,u,13.0,u,high,Answer (template filled):\n\ntumor 1: type = P...,INDICATION:\r\nCT ABDOMEN/PELVIS WITH AND WITH...,...,NaN,liver,yes,u,BDMAP_00085075,NaN,NaN,NaN,332.0,u
96847,NaN,tumor 3,liver,metastasis,u,multiple,u,high,Answer (template filled):\n\ntumor 1: type = P...,INDICATION:\r\nCT ABDOMEN/PELVIS WITH AND WITH...,...,NaN,liver,yes,u,BDMAP_00085075,NaN,NaN,NaN,332.0,u
96848,NaN,tumor 4,bone,metastasis,vertebral body of l1,22.0,u,high,Answer (template filled):\n\ntumor 1: type = P...,INDICATION:\r\nCT ABDOMEN/PELVIS WITH AND WITH...,...,NaN,bone,no,NaN,BDMAP_00085075,NaN,NaN,NaN,332.0,u
96849,NaN,tumor 5,bone,metastasis,right lower rib,11.0,u,high,Answer (template filled):\n\ntumor 1: type = P...,INDICATION:\r\nCT ABDOMEN/PELVIS WITH AND WITH...,...,NaN,bone,no,NaN,BDMAP_00085075,NaN,NaN,NaN,332.0,u


In [1]:
import pandas as pd
meta = pd.read_csv('/projects/bodymaps/Data/UCSFLLMOutputLarge27k.csv')

/tmp/ipykernel_1664871/1566170393.py:2: DtypeWarning: Columns (11,19,21) have mixed types. Specify dtype option on import or set low_memory=False.
  meta = pd.read_csv('/projects/bodymaps/Data/UCSFLLMOutputLarge27k.csv')


In [2]:
meta

,Encrypted Patient MRN,Tumor ID,Organ,Tumor Type,Tumor Location,Tumor Size (mm),Tumor Attenuation,Type Certainty,DNN Answer,Report,...,Standardized Attenuation,Standardized Organ,Unknow Tumor Size,Standardized Location,BDMAP_ID,Patient Age,Patient Sex,no lesion,Index,Mapped Attenuation
0,AQPHRXY1,tumor 1,kidney,u,perinephric,5.0,low,u,Answer (template filled):\n\ntumor 1: type = U...,INDICATION:\r\nCT ABDOMEN/PELVIS WITH CONTRAST...,...,low,kidney,no,u,BDMAP_00038206,69.0,Female,False,NaN,NaN
1,x0rEVlgI,tumor 1,kidney,u,u,9.0,hypodense,u,Answer (template filled):\n\ntumor 1: type = U...,INDICATION:\r\nCT ABDOMEN/PELVIS WITH CONTRAST...,...,low,kidney,no,u,BDMAP_00041303,66.0,Female,False,NaN,NaN
2,x0rEVlgI,tumor 2,spleen,cyst,u,u,hypodense,high,Answer (template filled):\n\ntumor 1: type = U...,INDICATION:\r\nCT ABDOMEN/PELVIS WITH CONTRAST...,...,low,spleen,yes,NaN,BDMAP_00041303,66.0,Female,False,NaN,NaN
3,ZLND5q9x,tumor 1,liver,metastasis,right lobe,u,hypoattenuating,low,Answer (template filled):\n\ntumor 1: type = m...,INTERPRETATION OF OUTSIDE [ADDRESS] ABDOMEN AN...,...,low,liver,yes,segment 5 / segment 6 / segment 7 / segment 8,NaN,62.0,Female,False,NaN,NaN
4,ZLND5q9x,tumor 2,liver,metastasis,right lobe,u,hypoattenuating,low,Answer (template filled):\n\ntumor 1: type = m...,INTERPRETATION OF OUTSIDE [ADDRESS] ABDOMEN AN...,...,low,liver,yes,segment 5 / segment 6 / segment 7 / segment 8,NaN,62.0,Female,False,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96846,NaN,tumor 2,liver,metastasis,u,13.0,u,high,Answer (template filled):\n\ntumor 1: type = P...,INDICATION:\r\nCT ABDOMEN/PELVIS WITH AND WITH...,...,NaN,liver,yes,u,BDMAP_00085075,NaN,NaN,NaN,332.0,u
96847,NaN,tumor 3,liver,metastasis,u,multiple,u,high,Answer (template filled):\n\ntumor 1: type = P...,INDICATION:\r\nCT ABDOMEN/PELVIS WITH AND WITH...,...,NaN,liver,yes,u,BDMAP_00085075,NaN,NaN,NaN,332.0,u
96848,NaN,tumor 4,bone,metastasis,vertebral body of l1,22.0,u,high,Answer (template filled):\n\ntumor 1: type = P...,INDICATION:\r\nCT ABDOMEN/PELVIS WITH AND WITH...,...,NaN,bone,no,NaN,BDMAP_00085075,NaN,NaN,NaN,332.0,u
96849,NaN,tumor 5,bone,metastasis,right lower rib,11.0,u,high,Answer (template filled):\n\ntumor 1: type = P...,INDICATION:\r\nCT ABDOMEN/PELVIS WITH AND WITH...,...,NaN,bone,no,NaN,BDMAP_00085075,NaN,NaN,NaN,332.0,u


In [3]:
meta[meta['no lesion'] == True]

,Encrypted Patient MRN,Tumor ID,Organ,Tumor Type,Tumor Location,Tumor Size (mm),Tumor Attenuation,Type Certainty,DNN Answer,Report,...,Standardized Attenuation,Standardized Organ,Unknow Tumor Size,Standardized Location,BDMAP_ID,Patient Age,Patient Sex,no lesion,Index,Mapped Attenuation
92288,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,INDICATION:\r\nCT ABDOMEN/PELVIS WITH CONTRAST...,...,NaN,NaN,NaN,NaN,BDMAP_00081037,40.0,Female,True,NaN,NaN
92289,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT ABDOMEN PEL W/CONTRAST [DATE]:42 PM\r\n\r\n...,...,NaN,NaN,NaN,NaN,BDMAP_00081203,57.0,Male,True,NaN,NaN
92290,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Renal protocol CT scan [DATE]\r\n\r\nClinical ...,...,NaN,NaN,NaN,NaN,BDMAP_00081349,65.0,Male,True,NaN,NaN
92291,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT ABDOMEN PEL W/CONTRAST [DATE]:52 AM\r\n\r\n...,...,NaN,NaN,NaN,NaN,BDMAP_00080388,49.0,Male,True,NaN,NaN
92292,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,INDICATION:\r\nCT ABDOMEN/PELVIS WITH CONTRAST...,...,NaN,NaN,NaN,NaN,BDMAP_00080078,33.0,Female,True,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94852,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT ABDOMEN/PELVIS WITH CONTRAST [DATE]:14 P...,...,NaN,NaN,NaN,NaN,BDMAP_00033916,NaN,NaN,True,NaN,NaN
94853,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT ABDOMEN/PELVIS WITH CONTRAST [DATE]:38 A...,...,NaN,NaN,NaN,NaN,BDMAP_00033917,NaN,NaN,True,NaN,NaN
94854,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT ABDOMEN/PELVIS WITH CONTRAST [DATE]:29 AM_...,...,NaN,NaN,NaN,NaN,BDMAP_00033918,NaN,NaN,True,NaN,NaN
94855,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CT ABDOMEN/PELVIS WITH CONTRAST [DATE]:03 AM_...,...,NaN,NaN,NaN,NaN,BDMAP_00033921,NaN,NaN,True,NaN,NaN


In [17]:
annotated_tumors = ['adrenal gland', 'bladder', 'colon', 'duodenum',
                    'esophagus', 'gallbladder','prostate','spleen','stomach']

def clean_ufo(reports):
    """
    This function gets a list of reports and removes cases of no interest:
    - We get the healthy patients
    - We get, for each tumor we have annotated organs, all reports that have known tumor size
    - We remove, for organs that have rignr and left (adrenal glands, kidneys), the reports that have unknown sub-segment (not right or left)
    Then, we print the number of useful cases per tumor
    """
    
    interest = {}
    
    for organ in annotated_tumors:
        interest[organ] = reports[reports['Standardized Organ'] == organ]
        interest[organ] = interest[organ][interest[organ]['Tumor Size (mm)'] != 'u']
        interest[organ] = interest[organ][interest[organ]['Tumor Size (mm)'] != 'multiple']
        interest[organ] = interest[organ][interest[organ]['Unknow Tumor Size'] == 'no']
        if organ in ['kidney','adrenal_gland','lung','breast','femur']:
            interest[organ] = interest[organ][interest[organ]['Standardized Location'].str.contains('right') | interest[organ]['Standardized Location'].str.contains('left')]
        print('Number of useful cases for %s: %s'%(organ, interest[organ]['BDMAP_ID'].nunique()))

    interest['healthy'] = reports[reports['no lesion'] == True]
    print('Number of healthy cases:', interest['healthy']['BDMAP_ID'].nunique())
    #concat
    interest = pd.concat(interest.values())
    interest = interest.drop_duplicates()
    print('Total number of useful cases:', interest['BDMAP_ID'].nunique())
    ids_of_interest = interest['BDMAP_ID'].unique().tolist()
    return interest, ids_of_interest

In [18]:
cleaned,ids = clean_ufo(meta)

Number of useful cases for adrenal gland: 1958
Number of useful cases for bladder: 929
Number of useful cases for colon: 672
Number of useful cases for duodenum: 419
Number of useful cases for esophagus: 155
Number of useful cases for gallbladder: 377
Number of useful cases for prostate: 473
Number of useful cases for spleen: 1475
Number of useful cases for stomach: 713
Number of healthy cases: 2569
Total number of useful cases: 9167


In [19]:
ids

['BDMAP_00038682',
 'BDMAP_00038831',
 'BDMAP_00044694',
 'BDMAP_00041758',
 'BDMAP_00041030',
 'BDMAP_00045068',
 'BDMAP_00044532',
 'BDMAP_00038022',
 'BDMAP_00038673',
 'BDMAP_00045133',
 'BDMAP_00038225',
 'BDMAP_00038870',
 'BDMAP_00044407',
 'BDMAP_00045595',
 'BDMAP_00040442',
 'BDMAP_00038717',
 'BDMAP_00045645',
 'BDMAP_00038792',
 'BDMAP_00045617',
 'BDMAP_00045373',
 'BDMAP_00038134',
 'BDMAP_00037179',
 'BDMAP_00039356',
 'BDMAP_00045886',
 'BDMAP_00046033',
 'BDMAP_00040887',
 'BDMAP_00041323',
 'BDMAP_00038176',
 'BDMAP_00036978',
 'BDMAP_00039647',
 nan,
 'BDMAP_00045631',
 'BDMAP_00038746',
 'BDMAP_00038362',
 'BDMAP_00045984',
 'BDMAP_00041756',
 'BDMAP_00041606',
 'BDMAP_00045729',
 'BDMAP_00037389',
 'BDMAP_00038604',
 'BDMAP_00041048',
 'BDMAP_00046042',
 'BDMAP_00037957',
 'BDMAP_00038223',
 'BDMAP_00045682',
 'BDMAP_00045688',
 'BDMAP_00037962',
 'BDMAP_00045932',
 'BDMAP_00045521',
 'BDMAP_00038076',
 'BDMAP_00038327',
 'BDMAP_00041408',
 'BDMAP_00045429',
 'BDMA

In [20]:
len(ids)

9168

In [25]:

#check pdac
import numpy as np
y = np.load('/projects/bodymaps/Pedro/data/JHH_lesion_types_medformer_npz/BDMAP_V0001442_gt.npz')['arr_0']

In [26]:
#labels:
pth = '/projects/bodymaps/Pedro/data/JHH_lesion_types_medformer/list/label_names.yaml'
import yaml
with open(pth, 'r') as f:
    classes = yaml.load(f, Loader=yaml.SafeLoader)
    #sort--we sorted when saving in nii2npy.py
    classes = sorted(classes)

In [27]:
y[classes.index('pancreatic_pdac')].sum()

99694

In [28]:
y.shape

(45, 483, 306, 351)

In [29]:
y[classes.index('pancreatic_lesion')].sum()

99694

In [33]:

#check pdac
import numpy as np
y = np.load('/projects/bodymaps/Pedro/data/AbdomenAtlas3_JHH_medformer_augmented_balanced_crops_lesion_types/BDMAP_V0001442_gt.npy')
y = np.unpackbits(y, axis=0)[:45]

In [34]:
y.shape

(45, 128, 128, 128)

In [35]:
y[classes.index('pancreatic_lesion')].sum()

75329

In [36]:
y[classes.index('pancreatic_pdac')].sum()

75329